## Huấn luyện mô hình Cropper
Trong notebook này, dùng để huấn luyện mô hình cropper tách NIC ra khỏi nền. Để tách NIC, chúng ta sử dụng mô hình segmentation trong thư viện detectron2.

Các bạn cần tham khảo để biết cách sử dụng thư viện này trước tại [đây](https://colab.research.google.com/drive/16jcaJoc6bCFAQ96jDe2HwtXj7BMD_-m5).

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="1"

import numpy as np
import json
import torch, torchvision
from PIL import ImageFile, Image, ImageOps
import shutil

ImageFile.LOAD_TRUNCATED_IMAGES = True

import detectron2
from detectron2.utils.logger import setup_logger

# import some common libraries
import numpy as np
import cv2
import random
import matplotlib.pyplot as plt
import onnx

# import some common detectron2 utilities
from detectron2 import model_zoo
from detectron2.engine import DefaultTrainer
from detectron2.engine import DefaultPredictor
from detectron2.evaluation import COCOEvaluator
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog
from detectron2.export import export_onnx_model, export_caffe2_model, add_export_config
from detectron2.utils.visualizer import ColorMode
from detectron2.structures import BoxMode
from imutils.perspective import order_points

plt.rcParams["figure.figsize"] = (20,20)

%matplotlib inline

Xác định vị trí của top-left corner thực tế 

In [ ]:
def top_left(coords):
    coords = [(x, y, i) for i, (x, y) in enumerate(coords)]
    pts = np.asarray(coords)
    
    xSorted = pts[np.argsort(pts[:, 0]), :]

    # grab the left-most and right-most points from the sorted
    # x-roodinate points
    leftMost = xSorted[:2, :]
    rightMost = xSorted[2:, :]

    # now, sort the left-most coordinates according to their
    # y-coordinates so we can grab the top-left and bottom-left
    # points, respectively
    leftMost = leftMost[np.argsort(leftMost[:, 1]), :]
    (tl, bl) = leftMost
    return tl

In [ ]:
def four_point_transform(image, pts):
    # obtain a consistent order of the points and unpack them
    # individually
    rect = pts
    (tl, tr, br, bl) = rect
    
    # compute the width of the new image, which will be the
    # maximum distance between bottom-right and bottom-left
    # x-coordiates or the top-right and top-left x-coordinates
    widthA = np.sqrt(((br[0] - bl[0]) ** 2) + ((br[1] - bl[1]) ** 2))
    widthB = np.sqrt(((tr[0] - tl[0]) ** 2) + ((tr[1] - tl[1]) ** 2))
    maxWidth = max(int(widthA), int(widthB))

    # compute the height of the new image, which will be the
    # maximum distance between the top-right and bottom-right
    # y-coordinates or the top-left and bottom-left y-coordinates
    heightA = np.sqrt(((tr[0] - br[0]) ** 2) + ((tr[1] - br[1]) ** 2))
    heightB = np.sqrt(((tl[0] - bl[0]) ** 2) + ((tl[1] - bl[1]) ** 2))
    maxHeight = max(int(heightA), int(heightB))

    # now that we have the dimensions of the new image, construct
    # the set of destination points to obtain a "birds eye view",
    # (i.e. top-down view) of the image, again specifying points
    # in the top-left, top-right, bottom-right, and bottom-left
    # order
    dst = np.array([
        [0, 0],
        [maxWidth - 1, 0],
        [maxWidth - 1, maxHeight - 1],
        [0, maxHeight - 1]], dtype="float32")
    
    # compute the perspective transform matrix and then apply it
    M = cv2.getPerspectiveTransform(rect, dst)
    warped = cv2.warpPerspective(image, M, (maxWidth, maxHeight))

    # return the warped image
    return warped

## Dataset 
Chuẩn bị dữ liệu để huấn luyện, cấu trúc theo chuẩn của thư viện detectron2, tham khảo thư viện này để biết chi tiết 

In [ ]:
def get_hssk_dicts(img_dir, json_file):
    with open(json_file) as f:
        imgs_anns = json.load(f)
        imgs_anns = imgs_anns['_via_img_metadata']
        
    dataset_dicts = []
    for idx, e in enumerate(list(imgs_anns.values())):
        try:
            record = {}
            objs = []

            filename = e['filename']        

            filename = os.path.join(img_dir, filename) 

            height, width = cv2.imread(filename, 0).shape[:2]

            record["file_name"] = filename
            record["image_id"] = idx
            record["height"] = height
            record["width"] = width
            regions = e["regions"]

            for region in regions:
                try:
                    shape_attributes = region['shape_attributes']
                    region_attributes = region['region_attributes']
                    shape_type = shape_attributes['name']

                    px = shape_attributes["all_points_x"]
                    py = shape_attributes["all_points_y"]
                    loai_giay_to = int(region_attributes['type'])                        

                    poly = [(x + 0.5, y + 0.5) for x, y in zip(px, py)]   
                    orientation = int(top_left(poly)[2])

                    poly = [p for x in poly for p in x]            


                    obj = {
                        "bbox": [np.min(px), np.min(py), np.max(px), np.max(py)],
                        "bbox_mode": BoxMode.XYXY_ABS,
                        "category_id": loai_giay_to*4 + orientation,
                        "segmentation": [poly],
                        "iscrowd": 0
                    }
                    objs.append(obj)

                except Exception as e:
                    print(regions, e)
                
        except Exception as e:    
            print(e)
            
        if len(objs) > 0:
            record["annotations"] = objs
            dataset_dicts.append(record)
            
    return dataset_dicts


In [ ]:
train_json_file = '/data2/sftp/vmg/upload/nic/du_lieu_ban_giao/nic/crop/train_annotation.json'
test_json_file = '/data2/sftp/vmg/upload/nic/du_lieu_ban_giao/nic/crop/test_annotation.json'
img_dir = '/data2/sftp/vmg/upload/nic/du_lieu_ban_giao/nic/crop/img'

meta = json.load(open(train_json_file))
meta = meta['_via_attributes']

classes = list(meta['region']['type']['options'].values())
classes = ['{}_{}'.format(c, i) for c in classes for i in range(4)]

In [ ]:
from detectron2.data import DatasetCatalog, MetadataCatalog

for d, fname in zip(["val", "train"], [test_json_file, train_json_file]):
    DatasetCatalog.register("hssk_" + d, lambda fname=fname: get_hssk_dicts(img_dir, fname))
    MetadataCatalog.get("hssk_" + d).set(thing_classes=classes)
        
hssk_metadata = MetadataCatalog.get("hssk_train")

In [ ]:
dataset_dicts = get_hssk_dicts(img_dir, test_json_file)

### Visualize test set

In [ ]:
for d in random.sample(dataset_dicts, 10):
    img = cv2.imread(d["file_name"])
    visualizer = Visualizer(img[:, :, ::-1], metadata=hssk_metadata, scale=0.5)    
    vis = visualizer.draw_dataset_dict(d)
    plt.figure(figsize = (20,20))
    plt.imshow(vis.get_image()[:, :, ::-1])

### Custom Trainer

In [ ]:
from detectron2.data import DatasetMapper, build_detection_train_loader, build_detection_test_loader
from detectron2.data import transforms as T
from detectron2.data import detection_utils as utils
import copy 

class CustomerDatasetMapper():
    def __init__(self, cfg, is_train):
        self.is_train = is_train
    
    def __call__(self, dataset_dict):
        
        # Implement a mapper, similar to the default DatasetMapper, but with your own customizations
        dataset_dict = copy.deepcopy(dataset_dict)  # it will be modified by code below
        image = utils.read_image(dataset_dict["file_name"], format="RGB")
        if self.is_train:
            image, transforms = T.apply_transform_gens([T.ResizeShortestEdge([1024, 1024]), T.RandomBrightness(0.9,1.1), T.RandomContrast(0.9, 1.1)], image)
        else:
            image, transforms = T.apply_transform_gens([T.ResizeShortestEdge([1024, 1024])], image)

        dataset_dict["image"] = torch.as_tensor(image.transpose(2, 0, 1).astype("float32"))

        annos = [
            utils.transform_instance_annotations(obj, transforms, image.shape[:2])
            for obj in dataset_dict.pop("annotations")
            if obj.get("iscrowd", 0) == 0
        ]
        instances = utils.annotations_to_instances(annos, image.shape[:2])
        dataset_dict["instances"] = utils.filter_empty_instances(instances)
        return dataset_dict

class MyTrainer(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, "inference")
        return COCOEvaluator(dataset_name, cfg, True, output_folder)
    
    @classmethod
    def build_train_loader(cls,cfg):
        return build_detection_train_loader(cfg, mapper=CustomerDatasetMapper(cfg, True))
    
    @classmethod
    def build_test_loader(cls, cfg, dataset_name):
        return build_detection_test_loader(cfg, dataset_name, mapper=CustomerDatasetMapper(cfg, False))

In [ ]:
cfg = get_cfg()
cfg.CLASSES = classes

cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
cfg.DATASETS.TRAIN = ("hssk_train",)
cfg.DATASETS.TEST = ("hssk_val",)

cfg.MODEL.WEIGHTS = '/var/www/html/nic/crop/model.pth'
cfg.SOLVER.IMS_PER_BATCH = 2
cfg.SOLVER.BASE_LR = 0.00025  
cfg.MODEL.BACKBONE.FREEZE_AT = 0


cfg.MODEL.ROI_HEADS.NUM_CLASSES = len(classes) 
cfg.TEST.EVAL_PERIOD = 7500
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5  # set the testing threshold for this model
cfg.INPUT.MIN_SIZE_TEST = 1024
cfg.INPUT.MAX_SIZE_TEST = 3000

In [ ]:
shutil.rmtree(cfg.OUTPUT_DIR, ignore_errors=True)
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

trainer = MyTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

In [ ]:
with open('config.yml', 'w') as f:
    f.write(cfg.dump())